# Position Bias Analysis

This notebook demonstrates comprehensive position bias detection in MCQ benchmarks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from metaeval.bias.detection import PositionBiasAnalyzer, detect_bias, get_accuracy_by_position
from metaeval.bias.stats.tests import chi_square_test, kruskal_wallis_test, cochrans_q_test
from metaeval.bias.stats.effects import cramers_v, kendalls_w

## Creating Test Data with Known Bias

Let's create synthetic data with controlled position bias to demonstrate detection:

In [ ]:
np.random.seed(42)

def create_biased_data(n_questions=200, n_models=4, bias_strength=0.2):
    """Create data with known position A bias."""
    positions = ["A", "B", "C", "D"]
    data = []
    
    for model_idx in range(n_models):
        model = f"Model_{chr(65 + model_idx)}"  # Model_A, Model_B, etc.
        base_accuracy = 0.5 + np.random.uniform(-0.1, 0.1)  # Random base accuracy
        
        for q_id in range(n_questions):
            for pos in positions:
                # Position A gets a boost (simulating bias)
                pos_bias = bias_strength if pos == "A" else 0.0
                prob_correct = base_accuracy + pos_bias
                correct = np.random.random() < prob_correct
                
                data.append({
                    "question_id": q_id,
                    "model": model,
                    "variant": pos,
                    "is_correct": int(correct),
                })
    
    return pd.DataFrame(data)

# Create data with moderate bias
biased_df = create_biased_data(bias_strength=0.15)
print(f"Dataset: {len(biased_df)} observations")
print(f"Models: {biased_df['model'].unique()}")

## Quick Bias Detection

In [ ]:
# Quick check for bias
for model in biased_df['model'].unique():
    has_bias = detect_bias(biased_df, model)
    accuracy = get_accuracy_by_position(biased_df, model)
    print(f"{model}: Bias detected = {has_bias}")
    print(f"  Accuracy: A={accuracy['A']:.3f}, B={accuracy['B']:.3f}, C={accuracy['C']:.3f}, D={accuracy['D']:.3f}")

## Detailed Analysis with PositionBiasAnalyzer

In [ ]:
analyzer = PositionBiasAnalyzer(biased_df)

# Get full report for one model
report = analyzer.analyze("Model_A")

print("="*60)
print(f"Position Bias Report: {report.model}")
print("="*60)

print("\nAccuracy by Position:")
for pos, acc in sorted(report.accuracy_by_position.items()):
    print(f"  Position {pos}: {acc:.4f}")

print("\nStatistical Tests:")
for test_name, result in report.test_results.items():
    print(f"  {test_name}:")
    print(f"    Statistic: {result['statistic']:.4f}")
    print(f"    p-value: {result['p_value']:.6f}")
    print(f"    Significant: {result['significant']}")

print("\nEffect Sizes:")
for effect_name, result in report.effect_sizes.items():
    print(f"  {effect_name}: {result['value']:.4f} ({result['interpretation']})")

## Visualizing Position Bias

In [ ]:
# Analyze all models
all_reports = analyzer.analyze_all_models()

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Accuracy by position for each model
positions = ["A", "B", "C", "D"]
x = np.arange(len(positions))
width = 0.2

for i, (model, report) in enumerate(all_reports.items()):
    accuracies = [report.accuracy_by_position[pos] for pos in positions]
    axes[0].bar(x + i*width, accuracies, width, label=model)

axes[0].set_xlabel('Answer Position')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy by Position Across Models')
axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(positions)
axes[0].legend()
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Chance')

# Plot 2: Effect sizes
models = list(all_reports.keys())
cramers_v_values = [all_reports[m].effect_sizes['cramers_v']['value'] for m in models]

axes[1].bar(models, cramers_v_values, color=['red' if r.has_significant_bias else 'green' for r in all_reports.values()])
axes[1].set_xlabel('Model')
axes[1].set_ylabel("Cramér's V")
axes[1].set_title('Position Bias Effect Size by Model')
axes[1].axhline(y=0.1, color='orange', linestyle='--', label='Small effect')
axes[1].axhline(y=0.3, color='red', linestyle='--', label='Medium effect')
axes[1].legend()

plt.tight_layout()
plt.show()

## Statistical Tests Explained

Metaeval uses multiple statistical tests to detect position bias:

1. **Chi-square test**: Tests if accuracy differs significantly across positions
2. **Kruskal-Wallis test**: Non-parametric alternative when assumptions are violated
3. **Cochran's Q test**: For related samples (same questions, different positions)

In [ ]:
# Direct use of statistical tests
model_data = biased_df[biased_df['model'] == 'Model_A']

# Pivot data for tests
pivot = model_data.pivot_table(
    index='question_id',
    columns='variant',
    values='is_correct',
    aggfunc='first'
).reset_index()

# Chi-square test
observed = model_data.groupby('variant')['is_correct'].agg(['sum', 'count'])
chi2_result = chi_square_test(observed['sum'].values, observed['count'].values)
print(f"Chi-square test: statistic={chi2_result.statistic:.4f}, p={chi2_result.p_value:.6f}")

# Cramér's V
v = cramers_v(observed['sum'].values, observed['count'].values)
print(f"Cramér's V: {v.value:.4f} ({v.interpretation})")

## Bootstrap Confidence Intervals

In [ ]:
# Bootstrap confidence intervals for accuracy difference
from metaeval.core.config import get_config

config = get_config()
print(f"Bootstrap iterations: {config.stats.bootstrap_iterations}")
print(f"Confidence level: {config.stats.confidence_level}")

# The full report includes bootstrap CIs
if 'confidence_intervals' in report.test_results:
    print("\nConfidence Intervals:")
    for pos, ci in report.test_results['confidence_intervals'].items():
        print(f"  {pos}: [{ci['lower']:.4f}, {ci['upper']:.4f}]")